# Diagnóstico Automatizado: IA no Estetoscópio Digital - Parte 2

### Instalação do Modelo spaCy para Português

In [1]:
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 16.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Configuração do spaCy e Tratamento de Negação

A Parte 2 utiliza a mesma estratégia de tratamento de negação aplicada na Parte 1, porém as funções necessárias são implementadas também neste notebook para permitir sua execução de forma independente.

O spaCy é utilizado para identificar o escopo sintático de expressões de negação, como `não`, `nunca`, `jamais`, `nem` e `tampouco`.

Os tokens identificados dentro desse escopo recebem o prefixo `NEG_` antes da vetorização com TF-IDF. Dessa forma, termos como `dor` e `NEG_dor` são tratados como características diferentes pelo classificador.

In [2]:
# ============================================================
# CONFIGURAÇÃO DO SPACY E TRATAMENTO DE NEGAÇÃO
# ============================================================

import spacy

# Carregamento do modelo em português
nlp = spacy.load("pt_core_news_sm")

print("[OK] Modelo spaCy em português carregado com sucesso.")


# Palavras utilizadas como indicadores de negação
CUES_NEGACAO = {
    "não",
    "nunca",
    "jamais",
    "nem",
    "tampouco"
}


def coletar_escopo_negacao(token, tokens_negados):
    """
    Percorre a árvore de dependências a partir do termo relacionado
    à negação e identifica os tokens pertencentes ao seu escopo.

    Verbos coordenados são interrompidos para evitar que a negação
    seja propagada para orações independentes.
    """

    tokens_negados.add(token.i)

    for filho in token.children:

        if filho.dep_ == "conj" and filho.pos_ in ("VERB", "AUX"):
            continue

        coletar_escopo_negacao(filho, tokens_negados)


def detectar_tokens_negados(doc):
    """
    Retorna os índices dos tokens que estão dentro do escopo
    de uma expressão de negação.
    """

    tokens_negados = set()

    for token in doc:

        if token.lemma_.lower() in CUES_NEGACAO:
            coletar_escopo_negacao(
                token.head,
                tokens_negados
            )

    return tokens_negados


def marcar_negacao(texto, nlp_model):
    """
    Marca tokens sob escopo de negação utilizando o prefixo NEG_.

    Exemplo:
        "Não sinto dor no peito"

    pode ser transformado em algo semelhante a:

        "NEG_Não NEG_sinto NEG_dor NEG_no NEG_peito"

    Isso permite que o TF-IDF diferencie sintomas afirmados
    de sintomas negados.
    """

    doc = nlp_model(texto)

    tokens_negados = detectar_tokens_negados(doc)

    partes = []

    for token in doc:

        if (
            token.i in tokens_negados
            and not token.is_punct
            and not token.is_space
        ):
            partes.append("NEG_" + token.text)

        else:
            partes.append(token.text)

    return " ".join(partes)

[OK] Modelo spaCy em português carregado com sucesso.


### Validação do Pré-processamento de Negação

Antes da criação do classificador, realizamos um teste simples para verificar se o tratamento de negação está funcionando corretamente.

In [3]:
frase_teste = "Não sinto dor no peito, apenas um cansaço leve."

frase_processada = marcar_negacao(
    frase_teste,
    nlp
)

print("Frase original:")
print(frase_teste)

print("\nFrase processada:")
print(frase_processada)

Frase original:
Não sinto dor no peito, apenas um cansaço leve.

Frase processada:
NEG_Não NEG_sinto NEG_dor NEG_no NEG_peito , NEG_apenas NEG_um NEG_cansaço NEG_leve .


# Parte 2 – Classificador básico de texto

Esta etapa constrói um **classificador supervisionado de risco** a partir de relatos simulados de pacientes. O objetivo é **simular, em contexto acadêmico, uma lógica simplificada de classificação de risco**, utilizando técnicas acessíveis de Processamento de Linguagem Natural e Machine Learning.
A Parte 2 está organizada em três blocos:

1. **Pré-processamento de negação** — utiliza análise de dependência sintática com spaCy para diferenciar sintomas afirmados de sintomas negados antes da vetorização.
2. **Pipeline de classificação** — utiliza TF-IDF + Regressão Logística, com avaliação por acurácia, matriz de confusão e `classification_report`.
3. **Auditoria e governança de dados** — analisa pesos, erros de classificação e possíveis distorções do dataset antes de realizar ajustes.

Embora a estratégia de tratamento de negação também seja utilizada na Parte 1, as funções necessárias são implementadas neste notebook para que a **Parte 2 possa ser executada de forma independente**.

### Pré-processamento de negação

Antes da vetorização, cada frase passa pela função `marcar_negacao()`, que prefixa tokens negados com `NEG_`.

Essa transformação é aplicada de forma consistente aos textos antes da vetorização, garantindo que a mesma regra de representação da negação seja utilizada nos conjuntos de treino e teste.

O vocabulário do TF-IDF é aprendido **exclusivamente a partir dos dados de treinamento** e posteriormente aplicado ao conjunto de teste, evitando que informações do teste participem do processo de aprendizagem.

# Classificação de Risco em Relatos de Pacientes (Treinamento)

### Objetivo

Esta etapa implementa um modelo de **classificação supervisionada de textos**, utilizando relatos simulados de pacientes para identificar automaticamente duas categorias de risco:

* **Alto risco**
* **Baixo risco**

O objetivo é demonstrar um fluxo básico de **Processamento de Linguagem Natural (NLP)** e **Machine Learning**, desde a preparação e validação do dataset até o treinamento, avaliação e teste do modelo com novos relatos.

---

### Métodos e conceitos utilizados

#### Dataset rotulado

O conjunto de dados inicial contém **50 frases simuladas**, sendo:

* 25 classificadas como `alto risco`;
* 25 classificadas como `baixo risco`.

A distribuição equilibrada das classes reduz possíveis distorções causadas por desbalanceamento.

Entre as 50 frases, 10 correspondem a **casos de negação**:

* 5 casos de negação total → `baixo risco`;
* 5 casos de negação parcial → `alto risco`.

Esses exemplos auxiliam na validação do pré-processamento de negação apresentado anteriormente.

### Separação entre treino e teste

Os dados são divididos em:

* **80% para treinamento**;
* **20% para teste**.

A divisão utiliza `stratify=y`, mantendo proporcionalmente as duas classes nos conjuntos de treinamento e teste.

O `random_state=42` permite reproduzir a mesma divisão dos dados durante novas execuções.

### Pré-processamento de negação

Antes da vetorização, cada frase passa pela função `marcar_negacao()`, que prefixa tokens identificados dentro do escopo de uma negação com `NEG_`.

Essa transformação é aplicada de forma consistente aos textos utilizados no treinamento e no teste.

O vocabulário do TF-IDF é aprendido **exclusivamente a partir dos dados de treinamento** e posteriormente aplicado ao conjunto de teste, evitando que informações do teste participem do processo de aprendizagem.

### TF-IDF

Os relatos dos pacientes são textos e precisam ser convertidos para uma representação numérica antes de serem utilizados pelo algoritmo de Machine Learning.

Para isso é utilizado o **TF-IDF (Term Frequency–Inverse Document Frequency)**.

O método atribui pesos aos termos considerando sua importância dentro dos documentos do conjunto de dados.

Também são utilizados:

* **unigramas:** palavras individuais;
* **bigramas:** combinações de duas palavras consecutivas;
* conversão para letras minúsculas;
* `sublinear_tf=True` para aplicar uma escala sublinear à frequência dos termos.

### Regressão Logística

A **Regressão Logística** é utilizada como algoritmo de classificação supervisionada.

Neste projeto, ela realiza uma **classificação binária**, aprendendo padrões presentes nos textos de treinamento para decidir entre:

* `alto risco`;
* `baixo risco`.

O modelo também fornece probabilidades associadas às classes, permitindo observar o grau de confiança de cada previsão.

---

## Avaliação do modelo

Após o treinamento, o modelo é avaliado utilizando os dados que não participaram do treinamento.

São utilizadas as seguintes métricas:

#### **Acurácia**

Indica a proporção de previsões realizadas corretamente em relação ao total de exemplos avaliados.

#### **Matriz de Confusão**

Permite observar classificações corretas e incorretas para as classes de alto e baixo risco.

#### **Classification Report**

Apresenta métricas detalhadas por classe:

* **Precision:** proporção das previsões de uma classe que realmente pertencem a ela;
* **Recall:** proporção dos exemplos reais de uma classe corretamente identificados;
* **F1-score:** combinação entre precision e recall.

---

### Teste com novos relatos

Após a avaliação, o modelo também é utilizado para classificar novas frases que não fazem parte do dataset de treinamento.

Os novos relatos passam pelo mesmo tratamento de negação antes da vetorização.

Também é calculada a probabilidade associada à classe prevista, permitindo observar a confiança do modelo em cada classificação.

---

### Verificação da qualidade do dataset

São realizadas verificações básicas de qualidade:

* quantidade total de registros;
* distribuição das classes;
* registros duplicados;
* valores ausentes.

---

### Fluxo do processo

**Dataset rotulado → Pré-processamento de negação → Separação treino/teste → TF-IDF → Regressão Logística → Previsões → Avaliação → Teste com novos relatos → Verificação do dataset**

### Propósito da etapa

Esta etapa concentra o **pipeline principal da Parte 2**, demonstrando o processo de construção de um classificador textual supervisionado desde a preparação da base até a análise inicial de seu comportamento.

In [4]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ==============================================================================
# 1. PREPARAÇÃO DA BASE DE DADOS ROTULADA (.csv)
# ==============================================================================
NOME_BASE_CSV = "base_triagem_risco.csv"

# Dataset simulado: 20 alto risco + 20 baixo risco + 5 negação total (baixo) + 5 negação parcial (alto)
dados_triagem = {
    "frase": [
        # --- Alto risco (20 originais) ---
        "Sinto uma dor intensa no peito que se espalha para o braço esquerdo",
        "Estou com uma pressão muito forte no tórax acompanhada de falta de ar",
        "Meu coração está disparando e tive tontura seguida de desmaio",
        "A falta de ar começou de repente e veio acompanhada de dor no peito",
        "Estou sentindo dor forte no tórax junto com suor frio e náusea",
        "Tenho uma pressão intensa no peito e formigamento no braço",
        "Sinto um aperto muito forte no peito e dificuldade para respirar",
        "Uma dor no peito começou durante o esforço físico e está muito intensa",
        "Estou com dificuldade grave para respirar e uma dor torácica súbita",
        "Tive uma tontura intensa acompanhada de dor no peito e visão turva",
        "Sinto uma pressão no peito que não melhora e estou com muita falta de ar",
        "Comecei a sentir uma dor forte no tórax acompanhada de fraqueza repentina",
        "Estou com dor intensa no peito e sensação de desmaio",
        "Sinto uma dor no peito que piora quando faço esforço e vem acompanhada de suor frio",
        "Minha respiração ficou muito difícil de repente e estou sentindo pressão no peito",
        "Tenho palpitações muito fortes acompanhadas de tontura e fraqueza",
        "Estou sentindo um aperto intenso no tórax e uma sensação de falta de ar",
        "A dor no peito é forte e começou junto com suor frio e enjoo",
        "Sinto dor torácica intensa acompanhada de falta de ar e tontura",
        "Uma forte pressão no peito começou repentinamente e estou com dificuldade para respirar",
        # --- Baixo risco (20 originais) ---
        "Estou cansado depois de um dia inteiro trabalhando",
        "Sinto uma dor nas costas depois de passar muitas horas sentado",
        "Tenho um desconforto muscular no peito depois de fazer exercícios",
        "Fico com sono e fadiga no período da tarde",
        "Estou com uma dor de cabeça que começou depois do almoço",
        "Sinto um pequeno incômodo nas articulações ao acordar",
        "Tenho sentido cansaço durante as últimas semanas, sem outros sintomas",
        "Estou com coriza e uma irritação leve na garganta",
        "Sinto um desconforto no estômago depois das refeições",
        "Tenho uma pontada na costela que desaparece rapidamente",
        "Estou cansado após caminhar por bastante tempo, mas consigo continuar normalmente",
        "Minha garganta está um pouco irritada desde ontem",
        "Sinto dores musculares depois de realizar atividades físicas",
        "Tenho um leve desconforto nas costas quando permaneço muito tempo sentado",
        "Estou com sono durante o dia porque dormi poucas horas na noite anterior",
        "Sinto uma pequena dor de cabeça que melhora quando descanso",
        "Tenho um desconforto abdominal ocasional depois de comer",
        "Estou sentindo uma dor muscular nas pernas após uma caminhada longa",
        "Minha garganta está irritada e estou com coriza leve",
        "Sinto um pequeno incômodo no ombro depois de carregar peso",
        # --- Negação total → baixo risco (5) ---
        "Não sinto dor no peito, apenas um leve cansaço",
        "Não tenho falta de ar nem palpitações",
        "Nunca senti aperto no tórax, somente um cansaço ocasional",
        "Jamais tive dor no peito ou desmaio",
        "Não sinto pressão no peito, apenas um desconforto leve nas costas",
        # --- Negação parcial → alto risco (5) ---
        "Não tenho falta de ar, mas sinto uma dor no peito intensa",
        "Não sinto tontura, porém tenho dor no peito que irradia para o braço",
        "Não tenho palpitações, mas estou com aperto no tórax e suor frio",
        "Não estou com desmaio, mas sinto dor torácica forte e falta de ar",
        "Não sinto fraqueza, porém tenho pressão no peito e dificuldade para respirar"
    ],
    "situacao": (
        ["alto risco"] * 20 +
        ["baixo risco"] * 20 +
        ["baixo risco"] * 5 +
        ["alto risco"] * 5
    )
}

# Criação/atualização incondicional da base
df_base = pd.DataFrame(dados_triagem)
df_base.to_csv(NOME_BASE_CSV, index=False, encoding="utf-8")
print(f"[OK] Base rotulada '{NOME_BASE_CSV}' criada/atualizada com {len(df_base)} frases.")

[OK] Base rotulada 'base_triagem_risco.csv' criada/atualizada com 50 frases.


### Ingestão e pré-processamento

A base rotulada é carregada do `.csv` e cada frase passa por `marcar_negacao()` **antes** da divisão treino/teste. Isso garante que o modelo veja, em treino e em teste, exatamente a mesma representação (tokens negados prefixados com `NEG_`).

A divisão estratificada mantém a proporção entre as classes nos dois conjuntos.

In [5]:
# ==============================================================================
# 2. INGESTÃO E DIVISÃO DOS DADOS (TREINO E TESTE)
# ==============================================================================
df = pd.read_csv(NOME_BASE_CSV, encoding="utf-8")

# Pré-processamento de negação: aplicado ANTES do split (consistência treino/teste)
df["frase"] = df["frase"].apply(lambda t: marcar_negacao(t, nlp))

X = df["frase"]
y = df["situacao"]

# Divisão estratificada (80% treino / 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"\n[INFO] Dados carregados: {len(df)} amostras totais.")
print(f"[INFO] Treino: {len(X_train)} frases | Teste: {len(X_test)} frases.")


[INFO] Dados carregados: 50 amostras totais.
[INFO] Treino: 40 frases | Teste: 10 frases.


In [6]:
# ==============================================================================
# 3. VETORIZAÇÃO E TREINAMENTO DO MODELO
# ==============================================================================
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

modelo = LogisticRegression(random_state=42)
modelo.fit(X_train_tfidf, y_train)
print("[OK] Modelo treinado com sucesso.")

# ==============================================================================
# 4. AVALIAÇÃO E MÉTRICAS DE DESEMPENHO
# ==============================================================================
y_pred = modelo.predict(X_test_tfidf)
acuracia = accuracy_score(y_test, y_pred)
matriz_confusao = confusion_matrix(y_test, y_pred, labels=["alto risco", "baixo risco"])

print("\n" + "=" * 80)
print("RELATÓRIO DE AVALIAÇÃO DO MODELO DE TRIAGEM")
print("=" * 80)
print(f"• Acurácia Global: {acuracia * 100:.2f}%\n")

print("• Matriz de Confusão:")
print("                   Predito: Alto | Predito: Baixo")
print(f"  Real: Alto Risco    {matriz_confusao[0][0]:^11} | {matriz_confusao[0][1]:^14}")
print(f"  Real: Baixo Risco   {matriz_confusao[1][0]:^11} | {matriz_confusao[1][1]:^14}\n")

print("• Relatório Detalhado por Classe:")
print(classification_report(y_test, y_pred))

# ==============================================================================
# 5. TESTE DE AUDITORIA E PREDIÇÃO EM NOVAS FRASES
# ==============================================================================
# As frases de teste também passam por `marcar_negacao()` antes de serem vetorizadas.
# Isso garante que o modelo as veja no mesmo formato com que foi treinado.
# Caso contrário, ele encontraria tokens desconhecidos (`dor`, `peito` sem `NEG_`) e classificaria incorretamente.

novas_frases_pacientes = [
    "Sinto um aperto no peito muito forte e suor frio há meia hora.",
    "Tive um leve cansaço nas pernas depois de caminhar bastante.",
    "Estou com dor de cabeça e garganta inflamada.",
    "Sinto falta de ar intensa ao ficar em pé e dor torácica.",
    # Novos casos de negação para auditoria
    "Não sinto dor no peito, apenas um cansaço leve.",
    "Não tenho falta de ar, mas sinto dor no peito forte."
]

print("=" * 80)
print("TESTE EM NOVOS RELATOS DE PACIENTES")
print("=" * 80)

# Pré-processamento de negação nos novos relatos
novas_frases_processadas = [marcar_negacao(f, nlp) for f in novas_frases_pacientes]

novos_vetores = vectorizer.transform(novas_frases_processadas)
novas_predicoes = modelo.predict(novos_vetores)
probabilidades = modelo.predict_proba(novos_vetores)
classes = modelo.classes_

for frase, pred, prob in zip(novas_frases_pacientes, novas_predicoes, probabilidades):
    idx_classe = list(classes).index(pred)
    confianca = prob[idx_classe] * 100
    print(f'Relato       : "{frase}"')
    print(f"Classificação: [{pred.upper()}] (Confiança: {confianca:.1f}%)")
    print("-" * 80)

# ==============================================================================
# 6. VERIFICAÇÃO DA QUALIDADE DO DATASET
# ==============================================================================
print("\n" + "=" * 80)
print("VERIFICAÇÃO DO DATASET")
print("=" * 80)
print(f"• Total de registros: {len(df)}")
print("\n• Distribuição das classes:")
print(df["situacao"].value_counts())
print(f"\n• Registros duplicados: {df.duplicated().sum()}")
print(f"• Valores ausentes:")
print(df.isnull().sum())


[OK] Modelo treinado com sucesso.

RELATÓRIO DE AVALIAÇÃO DO MODELO DE TRIAGEM
• Acurácia Global: 90.00%

• Matriz de Confusão:
                   Predito: Alto | Predito: Baixo
  Real: Alto Risco         5      |       0       
  Real: Baixo Risco        1      |       4       

• Relatório Detalhado por Classe:
              precision    recall  f1-score   support

  alto risco       0.83      1.00      0.91         5
 baixo risco       1.00      0.80      0.89         5

    accuracy                           0.90        10
   macro avg       0.92      0.90      0.90        10
weighted avg       0.92      0.90      0.90        10

TESTE EM NOVOS RELATOS DE PACIENTES
Relato       : "Sinto um aperto no peito muito forte e suor frio há meia hora."
Classificação: [ALTO RISCO] (Confiança: 58.6%)
--------------------------------------------------------------------------------
Relato       : "Tive um leve cansaço nas pernas depois de caminhar bastante."
Classificação: [BAIXO RISCO] (Confia

# Análise dos Pesos da Regressão Logística

Nesta etapa, examinamos os **pesos atribuídos pela Regressão Logística aos termos do TF-IDF** para entender quais palavras e expressões influenciam as decisões do modelo.

O objetivo é verificar se o modelo está aprendendo **padrões coerentes com as classes de risco** ou se existem **distorções causadas pela composição do dataset** — por exemplo, termos ambíguos com peso excessivo ou prefixos `NEG_` ainda pouco representados.

Em vez de buscar imediatamente aumentar a acurácia, optamos primeiro por **auditar o comportamento do modelo**. Essa abordagem permite identificar relações artificiais aprendidas antes de ajustar algoritmo ou hiperparâmetros.

In [7]:
# ==============================================================================
# 7. ANÁLISE DOS PESOS DA REGRESSÃO LOGÍSTICA
# ==============================================================================
termos = vectorizer.get_feature_names_out()
coeficientes = modelo.coef_[0]

df_pesos = pd.DataFrame({"termo": termos, "peso": coeficientes})
df_pesos_ordenado = df_pesos.sort_values(by="peso", ascending=False)

print("\n" + "=" * 80)
print("ANÁLISE DOS TERMOS QUE INFLUENCIAM A CLASSIFICAÇÃO")
print("=" * 80)

# Termos com maior peso positivo (baixo risco)
print("\n→ Termos que mais favorecem a classe 'baixo risco':")
print(df_pesos_ordenado[["termo", "peso"]].head(15).to_string(index=False))

# Termos com maior peso negativo (alto risco)
print("\n→ Termos que mais favorecem a classe 'alto risco':")
print(df_pesos_ordenado[["termo", "peso"]].tail(15).sort_values(by="peso").to_string(index=False))

# ==============================================================================
# 8. VISUALIZAÇÃO DOS TERMOS MAIS INFLUENTES
# ==============================================================================
print("\n" + "=" * 80)
print("INTERPRETAÇÃO")
print("=" * 80)
print("""
Na Regressão Logística:

• Peso positivo  → favorece a classe representada pelo coeficiente positivo.
• Peso negativo  → favorece a outra classe.
• Quanto maior o valor absoluto do peso, maior a influência daquele termo
  na decisão do modelo.

A análise permite identificar quais palavras e expressões aprendidas pelo
TF-IDF estão influenciando as classificações realizadas pelo modelo.
""")


ANÁLISE DOS TERMOS QUE INFLUENCIAM A CLASSIFICAÇÃO

→ Termos que mais favorecem a classe 'baixo risco':
         termo     peso
        depois 0.447380
     depois de 0.349608
            um 0.309498
           nas 0.279485
          leve 0.237141
   desconforto 0.233852
      sinto um 0.232853
      garganta 0.185609
    com coriza 0.185609
        coriza 0.185609
        cabeça 0.183721
    cabeça que 0.183721
        dor de 0.183721
     de cabeça 0.183721
um desconforto 0.182036

→ Termos que mais favorecem a classe 'alto risco':
         termo      peso
      no peito -0.458940
         peito -0.458940
            no -0.427450
       intensa -0.400872
           dor -0.370193
        dor no -0.363841
   acompanhada -0.358242
acompanhada de -0.358242
         forte -0.329988
         falta -0.318464
         de ar -0.318464
            ar -0.318464
      falta de -0.318464
       tontura -0.311442
          para -0.295007

INTERPRETAÇÃO

Na Regressão Logística:

• Peso positivo  →

# Análise dos Erros de Classificação

Após auditar os pesos, a próxima decisão é investigar **quais registros foram classificados incorretamente** pelo modelo.

A análise dos erros permite relacionar o comportamento observado nos pesos com as previsões realizadas, identificando possíveis padrões que estejam causando decisões inadequadas.

Neste momento, optamos por **não alterar o algoritmo nem buscar aumentar a acurácia imediatamente**. Primeiro analisamos os erros para verificar se estão relacionados a:

- pouca quantidade de dados;
- semelhança entre frases de classes diferentes;
- presença de termos ambíguos;
- ou padrões artificiais existentes no dataset.

Essa abordagem permite melhorar a qualidade dos dados e a capacidade de generalização do modelo de forma fundamentada, em vez de apenas otimizar uma métrica.

> **Objetivo da análise:** compreender o comportamento do modelo antes de realizar ajustes no dataset ou nos parâmetros do classificador.

In [8]:
# ==============================================================================
# 9. ANÁLISE DOS ERROS DE CLASSIFICAÇÃO
# ==============================================================================
print("\n" + "=" * 80)
print("ANÁLISE DOS ERROS DE CLASSIFICAÇÃO")
print("=" * 80)

# Tabela com dados reais e previsões do modelo
df_analise_erros = pd.DataFrame({
    "frase": X_test.values,
    "classe_real": y_test.values,
    "classe_predita": y_pred
})

# Identificação das classificações incorretas
erros = df_analise_erros[df_analise_erros["classe_real"] != df_analise_erros["classe_predita"]]

print(f"\n[INFO] Total de exemplos avaliados: {len(df_analise_erros)}")
print(f"[INFO] Total de classificações incorretas: {len(erros)}")

if len(erros) == 0:
    print("\n[OK] Nenhum erro de classificação foi encontrado no conjunto de teste.")
else:
    print("\n[ATENÇÃO] Exemplos classificados incorretamente:\n")
    for _, linha in erros.iterrows():
        print(f"Frase          : {linha['frase']}")
        print(f"Classe real    : {linha['classe_real']}")
        print(f"Classe prevista: {linha['classe_predita']}")
        print("-" * 80)


ANÁLISE DOS ERROS DE CLASSIFICAÇÃO

[INFO] Total de exemplos avaliados: 10
[INFO] Total de classificações incorretas: 1

[ATENÇÃO] Exemplos classificados incorretamente:

Frase          : NEG_Não NEG_tenho NEG_falta NEG_de NEG_ar NEG_nem NEG_palpitações
Classe real    : baixo risco
Classe prevista: alto risco
--------------------------------------------------------------------------------


In [9]:
# ==============================================================================
# 10. INVESTIGAÇÃO DO FALSO POSITIVO
# ==============================================================================
print("\n" + "=" * 80)
print("INVESTIGAÇÃO DO FALSO POSITIVO")
print("=" * 80)

# Frase classificada incorretamente
frase_erro = erros.iloc[0]["frase"]
print(f"\nFrase analisada:\n\"{frase_erro}\"")

# Vetorização e extração de termos
vetor_frase = vectorizer.transform([frase_erro])
indices = vetor_frase.nonzero()[1]

analise_termos = pd.DataFrame({
    "termo": vectorizer.get_feature_names_out()[indices],
    "tfidf": vetor_frase.toarray()[0][indices],
    "peso_modelo": modelo.coef_[0][indices]
})

# Contribuição dos termos
analise_termos["contribuicao"] = analise_termos["tfidf"] * analise_termos["peso_modelo"]
analise_termos = analise_termos.sort_values(by="contribuicao", ascending=False)

print("\nTermos que mais contribuíram para a decisão:")
print(analise_termos.to_string(index=False))
print("\n" + "=" * 80)


INVESTIGAÇÃO DO FALSO POSITIVO

Frase analisada:
"NEG_Não NEG_tenho NEG_falta NEG_de NEG_ar NEG_nem NEG_palpitações"

Termos que mais contribuíram para a decisão:
              termo    tfidf  peso_modelo  contribuicao
             neg_ar 0.331664    -0.092283     -0.030607
             neg_de 0.331664    -0.092283     -0.030607
      neg_de neg_ar 0.331664    -0.092283     -0.030607
          neg_falta 0.331664    -0.092283     -0.030607
   neg_falta neg_de 0.331664    -0.092283     -0.030607
neg_tenho neg_falta 0.331664    -0.092283     -0.030607
    neg_palpitações 0.331664    -0.099672     -0.033058
  neg_não neg_tenho 0.298215    -0.172597     -0.051471
          neg_tenho 0.298215    -0.172597     -0.051471
            neg_não 0.228318    -0.250738     -0.057248



# Ajuste do Dataset e Redução de Distorções

#### Objetivo

A análise do falso positivo mostrou que o modelo classificou como **alto risco** uma frase com **negação total** ("Não tenho falta de ar nem palpitações"), apesar do prefixo `NEG_` aplicado aos tokens negados.

Isso indica que, embora o pré-processamento de negação esteja correto, o **vocabulário de tokens negados ainda é pouco representado no conjunto de treino** — o modelo aprendeu os padrões de tokens afirmados com mais força do que os negados.

#### Decisão metodológica

Em vez de substituir imediatamente o algoritmo, optamos por **melhorar a representatividade do dataset**.

O objetivo é fazer com que o modelo encontre padrões mais contextualizados, evitando depender excessivamente de palavras isoladas. Para isso, serão incluídas novas frases que apresentem diferentes contextos para termos potencialmente ambíguos, como **dor**, **cansaço** e **falta de ar**.

Essa abordagem busca melhorar a capacidade de generalização do modelo e reduzir falsos positivos causados por associações artificiais presentes em uma base pequena.

Após o ajuste, o modelo será treinado novamente e sua acurácia e matriz de confusão serão comparadas com os resultados anteriores.


> **Nota:** as frases de negação adicionadas na Seção 1 já contemplam parte do viés de "termo isolado → alto risco". Esta seção complementa o ajuste com frases contextuais para os termos "dor", "cansaço" e "falta de ar" em contextos ambíguos, ampliando também a variedade de contextos afirmativos que o modelo observa no treino.

In [10]:
# ==============================================================================
# 11. AJUSTE DO DATASET
# ==============================================================================

novos_dados = pd.DataFrame({
    "frase": [
        "Estou com uma dor muscular nas pernas depois de uma caminhada",
        "Sinto dor muscular nos braços após realizar exercícios",
        "Tenho uma dor leve nas pernas depois de uma atividade física",
        "Estou sentindo dor muscular nas costas após carregar peso",
        "Tenho um desconforto muscular nas pernas que melhora com descanso",
        "Estou com uma dor intensa no peito acompanhada de falta de ar",
        "Sinto uma dor forte no tórax que começou repentinamente",
        "Tenho uma dor intensa no peito acompanhada de suor frio",
        "Estou com dor torácica forte e sensação de desmaio",
        "Sinto uma dor muito forte no peito durante esforço físico"
    ],
    "situacao": [
        "baixo risco",
        "baixo risco",
        "baixo risco",
        "baixo risco",
        "baixo risco",
        "alto risco",
        "alto risco",
        "alto risco",
        "alto risco",
        "alto risco"
    ]
})

# Une a base ORIGINAL, ainda sem pré-processamento,
# aos novos exemplos contextualizados.
df_atualizada = pd.concat(
    [df_base, novos_dados],
    ignore_index=True
).drop_duplicates(subset=["frase"])

# Salva as frases em formato original.
# O tratamento de negação será reaplicado no retreinamento.
df_atualizada.to_csv(
    NOME_BASE_CSV,
    index=False,
    encoding="utf-8"
)

print("[OK] Dataset atualizado com sucesso.")
print(f"[INFO] Total de registros: {len(df_atualizada)}")

print("\n[INFO] Distribuição das classes:")
print(df_atualizada["situacao"].value_counts())

[OK] Dataset atualizado com sucesso.
[INFO] Total de registros: 60

[INFO] Distribuição das classes:
situacao
alto risco     30
baixo risco    30
Name: count, dtype: int64


# Retreinamento e Reavaliação do Modelo

Objetivo

Após a inclusão de exemplos mais diversificados no dataset, o modelo será treinado novamente para verificar se passou a reconhecer melhor o contexto das frases.

Decisão metodológica

Será mantida a mesma abordagem utilizada anteriormente — divisão treino/teste, TF-IDF e Regressão Logística — alterando inicialmente apenas os dados de treinamento.

Dessa forma, podemos comparar os resultados antes e depois da melhoria do dataset sem introduzir simultaneamente mudanças no algoritmo.

O pipeline de pré-processamento também é reaplicado integralmente: o dataset atualizado é recarregado do `.csv` e cada frase passa novamente por `marcar_negacao()` antes do `train_test_split`. Isso garante que o modelo retreinado utilize exatamente a mesma representação do modelo inicial (tokens sob negação prefixados com `NEG_`), mantendo a comparação antes × depois válida.

Serão novamente avaliados:

- acurácia;
- matriz de confusão;
- precision, recall e F1-score;
- comportamento diante de novos relatos.

O objetivo não é apenas obter uma acurácia maior, mas verificar se houve redução do falso positivo identificado anteriormente e se o modelo passou a utilizar padrões mais contextualizados — incluindo o tratamento correto de sintomas negados.

In [11]:
# ==============================================================================
# 12. RETREINAMENTO E REAVALIAÇÃO DO MODELO
# ==============================================================================
df = pd.read_csv(NOME_BASE_CSV, encoding="utf-8")

# Pré-processamento de negação (mesma função aplicada na Seção 2)
df["frase"] = df["frase"].apply(lambda t: marcar_negacao(t, nlp))

X, y = df["frase"], df["situacao"]

# Separação e vetorização TF-IDF
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True, sublinear_tf=True)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Treinamento e Predição
modelo = LogisticRegression(random_state=42)
modelo.fit(X_train_tfidf, y_train)
y_pred = modelo.predict(X_test_tfidf)
acuracia = accuracy_score(y_test, y_pred)

print("\n" + "=" * 80)
print("REAVALIAÇÃO DO MODELO APÓS AJUSTE DO DATASET")
print("=" * 80)
print(f"\n• Registros utilizados: {len(df)}")
print(f"• Acurácia: {acuracia * 100:.2f}%")
print("\n• Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred, labels=["alto risco", "baixo risco"]))
print("\n• Relatório de Classificação:")
print(classification_report(y_test, y_pred))


REAVALIAÇÃO DO MODELO APÓS AJUSTE DO DATASET

• Registros utilizados: 60
• Acurácia: 100.00%

• Matriz de Confusão:
[[6 0]
 [0 6]]

• Relatório de Classificação:
              precision    recall  f1-score   support

  alto risco       1.00      1.00      1.00         6
 baixo risco       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



# Comparação dos Resultados Antes e Depois do Ajuste

## Objetivo

Comparar o desempenho do modelo antes e depois da melhoria do dataset, verificando se as alterações contribuíram para reduzir o erro identificado anteriormente.

## Resultado observado

O modelo inicial, treinado com **50 frases**, apresentou acurácia de **90,00%** no conjunto de teste.

Após a investigação do falso positivo, o dataset foi ampliado para **60 frases**, mantendo o equilíbrio entre as classes:

* 30 exemplos de `alto risco`;
* 30 exemplos de `baixo risco`.

O modelo foi então treinado novamente mantendo a mesma metodologia de pré-processamento, TF-IDF e Regressão Logística.

Na avaliação final, foram utilizadas **12 amostras de teste**, correspondentes a 20% do dataset atualizado.

## Interpretação

A comparação permite avaliar o impacto da melhoria da representatividade do dataset sem alterar simultaneamente o algoritmo utilizado.

Os novos exemplos foram adicionados para fornecer maior variedade de contexto a expressões potencialmente ambíguas, além de manter exemplos envolvendo negação.

O resultado deve ser interpretado considerando que o dataset é **pequeno, simulado e controlado**. Portanto, mesmo uma acurácia elevada não representa garantia de desempenho em dados clínicos reais.

In [12]:
# ==============================================================================
# 13. COMPARAÇÃO ANTES × DEPOIS
# ==============================================================================

acuracia_anterior = 0.90
acuracia_atual = acuracia
melhoria = (acuracia_atual - acuracia_anterior) * 100

matriz_atual = confusion_matrix(
    y_test,
    y_pred,
    labels=["alto risco", "baixo risco"]
)

falsos_negativos = matriz_atual[0][1]
falsos_positivos = matriz_atual[1][0]

print("\n" + "=" * 80)
print("COMPARAÇÃO DOS RESULTADOS")
print("=" * 80)

print(f"\n• Acurácia anterior : {acuracia_anterior * 100:.2f}%")
print(f"• Acurácia atual    : {acuracia_atual * 100:.2f}%")
print(f"• Melhoria          : {melhoria:+.2f} pontos percentuais")

print("\n• Resultado da avaliação atual:")
print(f"  - Falsos positivos: {falsos_positivos}")
print(f"  - Falsos negativos: {falsos_negativos}")
print(f"  - Amostras de teste: {len(y_test)}")


COMPARAÇÃO DOS RESULTADOS

• Acurácia anterior : 90.00%
• Acurácia atual    : 100.00%
• Melhoria          : +10.00 pontos percentuais

• Resultado da avaliação atual:
  - Falsos positivos: 0
  - Falsos negativos: 0
  - Amostras de teste: 12


### Teste interativo

O relato digitado pelo usuário passa por `marcar_negacao()` antes de ser vetorizado — mesma função usada no treino e no teste. Isso permite testar frases com ou sem negação e observar o classificador responder de forma consistente.

**Sugestões de teste:**
- `Não sinto dor no peito, apenas um cansaço leve` → esperado: baixo risco
- `Não tenho falta de ar, mas sinto dor no peito forte` → esperado: alto risco
- `Sinto aperto no peito e suor frio` → esperado: alto risco

Frases sem negação também são aceitas normalmente — o pré-processamento apenas marca os tokens negados quando eles existem, deixando o restante do texto inalterado.

A entrada é limitada a **100 caracteres** para manter o teste objetivo e adequado à visualização na interface.

> **Importante:** esta funcionalidade é uma demonstração acadêmica de Machine Learning. A classificação não representa diagnóstico médico nem deve ser utilizada para tomada de decisão clínica.

In [13]:
# ==============================================================================
# 14. TESTE INTERATIVO DO CLASSIFICADOR
# ==============================================================================
print("\n" + "=" * 80)
print("TESTE INTERATIVO DO CLASSIFICADOR")
print("=" * 80)

relato = input("\nDigite seu relato (máximo 100 caracteres): ").strip()

if len(relato) > 100:
    print("\n[ERRO] O relato deve possuir no máximo 100 caracteres.")
elif not relato:
    print("\n[ERRO] O relato não pode estar vazio.")
else:
    # Pré-processamento de negação (mesma função usada no treino)
    relato_processado = marcar_negacao(relato, nlp)

    vetor = vectorizer.transform([relato_processado])
    predicao = modelo.predict(vetor)[0]
    probabilidades = modelo.predict_proba(vetor)[0]
    confianca = probabilidades[list(modelo.classes_).index(predicao)] * 100

    print("\n" + "-" * 80)
    print(f"Relato       : {relato}")
    print(f"Classificação: [{predicao.upper()}]")
    print(f"Confiança    : {confianca:.1f}%")
    print("-" * 80)
    print("\n[SIMULAÇÃO ACADÊMICA] O resultado não representa um diagnóstico médico.")


TESTE INTERATIVO DO CLASSIFICADOR

Digite seu relato (máximo 100 caracteres): Não sinto dor no peito, apenas um cansaço leve

--------------------------------------------------------------------------------
Relato       : Não sinto dor no peito, apenas um cansaço leve
Classificação: [BAIXO RISCO]
Confiança    : 67.3%
--------------------------------------------------------------------------------

[SIMULAÇÃO ACADÊMICA] O resultado não representa um diagnóstico médico.


# Governança, Auditoria de Vieses e Justiça Algorítmica

## Por que este tema importa

Sistemas de IA aplicados à saúde já mostraram vieses documentados na literatura — por exemplo, algoritmos de triagem que subestimam dor em mulheres ou que têm desempenho inferior em populações minoritárias. Antes de qualquer implantação real, um sistema como o desta atividade precisaria passar por auditoria de fairness.

## O que uma auditoria de vieses exigiria

1. **Atributos protegidos** — idade, sexo, raça, região, comorbidades
2. **Métricas de paridade** — Demographic Parity, Equal Opportunity, Disparate Impact
3. **Validação com comitê de ética** — aprovação institucional
4. **Monitoramento contínuo** — o modelo pode enviesar com o tempo
5. **Transparência com o paciente** — direito à explicação

## Por que não foi implementada nesta atividade

O dataset desta atividade é **simulado** e **não contém atributos demográficos**. Sem grupos protegidos, não é possível medir disparidades reais.

Ainda assim, o pipeline construído oferece o ponto de partida: a análise de pesos (Seção 7) e a investigação de erros (Seção 9) são etapas precursoras de uma auditoria. Se o dataset fosse ampliado com metadados reais, bastaria agrupar por atributo e comparar as métricas por subgrupo.

## Boas práticas adotadas neste projeto

- **Aviso ético** em todos os outputs ("simulação acadêmica, não diagnóstico médico")
- **Transparência do modelo** — análise de pesos e explicação das decisões
- **Auditoria de erros** — investigação individual do falso positivo em vez de só otimizar acurácia
- **Documentação de limitações** — reconhecimento explícito do que o sistema não cobre

Esses são os primeiros passos de uma postura de IA responsável, alinhados ao Cap. 07 do material da fase.

# Conclusão da Atividade

## Conclusão

A Parte 2 permitiu desenvolver um **classificador supervisionado de risco** utilizando técnicas clássicas de Processamento de Linguagem Natural e Machine Learning.

Os relatos simulados foram submetidos a um pré-processamento para tratamento de negação, representados numericamente por meio de **TF-IDF** e classificados utilizando **Regressão Logística**.

Durante o desenvolvimento, o comportamento do modelo foi analisado por meio das métricas de avaliação, dos pesos aprendidos pela Regressão Logística e da investigação individual dos erros de classificação.

A identificação de um falso positivo motivou uma revisão da base de dados. Novos exemplos contextualizados foram adicionados, ampliando o dataset de **50 para 60 frases**, mantendo o equilíbrio entre as classes de `alto risco` e `baixo risco`.

Após o ajuste, o modelo foi novamente treinado utilizando a mesma metodologia, permitindo comparar os resultados antes e depois da melhoria dos dados.

## Principais Entregáveis

* Dataset `.csv` de classificação de risco;
* base final com **60 frases balanceadas**;
* tratamento de negação com spaCy;
* representação textual utilizando **TF-IDF**;
* classificador utilizando **Regressão Logística**;
* avaliação com acurácia, matriz de confusão, precision, recall e F1-score;
* análise dos pesos aprendidos pelo modelo;
* investigação de erros de classificação;
* ajuste e ampliação do dataset;
* retreinamento e reavaliação do modelo;
* teste interativo com novos relatos.

## Desafios Encontrados

Um dos principais desafios foi identificar como determinadas palavras e expressões influenciavam a classificação.

A auditoria revelou um falso positivo envolvendo uma frase com sintomas negados. Em vez de alterar imediatamente o algoritmo, foi realizada uma análise dos termos e dos pesos aprendidos pelo modelo.

Também foi necessário tratar explicitamente a **negação**, pois o TF-IDF, isoladamente, não interpreta o significado contextual de expressões como `"não sinto dor no peito"`.

Para isso, tokens localizados dentro do escopo de uma negação recebem o prefixo `NEG_`, permitindo que o vetorizador diferencie sintomas afirmados de sintomas negados.

## Práticas Implementadas

O dataset foi ampliado com exemplos mais contextualizados e permaneceu equilibrado entre as duas classes.

O mesmo pré-processamento é utilizado durante treinamento, teste e classificação de novos relatos, garantindo consistência na representação dos textos.

Também foram realizadas auditoria dos pesos, investigação dos erros e comparação dos resultados antes e depois do ajuste do dataset.

> **Observação:** este projeto possui finalidade exclusivamente acadêmica. As classificações apresentadas não representam diagnóstico, orientação médica ou sistema clínico validado.